# Módulo 8 · Clase 4 — TensorFlow y Keras: Arquitectura Básica

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | ¿Qué es TensorFlow y Keras? |
| 2 | Tensores: el dato fundamental |
| 3 | API Sequential: construir modelos |
| 4 | Compilar y entrenar |
| 5 | Evaluar, predecir y guardar |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU disponibles: {len(gpus)} {'(entrenamiento acelerado)' if gpus else '(CPU mode)'}")

---
## 1. ¿Qué es TensorFlow y Keras?

```
TENSORFLOW (Google Brain, 2015):
  Framework de cómputo numérico para ML/DL.
  Calcula AUTOMÁTICAMENTE los gradientes (diferenciación automática).
  Ejecuta operaciones en CPU, GPU o TPU sin cambiar el código.
  Librería de bajo nivel → control total, pero verboso.

KERAS (François Chollet, 2015 → dentro de TF desde 2.0):
  API de alto nivel sobre TensorFlow.
  Diseñado para ser simple y legible.
  Permite construir redes en pocas líneas.
  "Deep learning for humans"

Relación:
  TensorFlow  →  el motor
  Keras       →  el volante y los pedales
  import tensorflow as tf
  from tensorflow import keras

Alternativas:
  PyTorch (Meta): más popular en investigación
  JAX (Google): para investigación de vanguardia
  sklearn: para ML clásico (no DL)

Cuándo usar TF/Keras:
  → producción, APIs web, mobile, hardware especializado
  → proyectos con datos de imagen, audio, texto
  → cuando quieres código estable y bien documentado
```

---
## 2. Tensores: el dato fundamental

In [ ]:
# Los tensores son como arrays NumPy pero con soporte GPU y gradientes automáticos

# Crear tensores
escalar   = tf.constant(3.14)                          # 0D: un número
vector    = tf.constant([1.0, 2.0, 3.0])               # 1D: vector
matriz    = tf.constant([[1.0, 2.0], [3.0, 4.0]])       # 2D: matriz
tensor_3d = tf.constant(np.random.randn(2, 3, 4))       # 3D: batch de matrices

print("Tensores:")
for nombre, t in [('escalar',escalar),('vector',vector),('matriz',matriz),('3D',tensor_3d)]:
    print(f"  {nombre:10}: shape={t.shape}, dtype={t.dtype.name}")

# Operaciones (igual que NumPy)
a = tf.constant([1.0, 2.0, 3.0])
b = tf.constant([4.0, 5.0, 6.0])
print(f"\na + b     = {(a+b).numpy()}")
print(f"a * b     = {(a*b).numpy()}")
print(f"a @ b T   = {tf.tensordot(a, b, axes=1).numpy():.1f}")
print(f"tf.reduce_mean = {tf.reduce_mean(a).numpy():.1f}")

# Convertir a NumPy
t = tf.constant([[1.0, 2.0], [3.0, 4.0]])
arr = t.numpy()
print(f"\nTensor → NumPy: {type(arr).__name__}, shape={arr.shape}")

In [ ]:
# Diferenciación automática: TF calcula gradientes sin backprop manual
w = tf.Variable(0.5)   # Variable: parámetro que TF puede derivar
x = tf.constant(2.0)
y = tf.constant(3.0)   # target

with tf.GradientTape() as tape:
    y_hat = w * x           # predicción
    loss  = (y_hat - y)**2  # MSE de 1 muestra

grad = tape.gradient(loss, w)  # ∂loss/∂w automáticamente

print("Diferenciación automática:")
print(f"  w={w.numpy()}, x={x.numpy()}, y={y.numpy()}")
print(f"  y_hat = w·x = {(w*x).numpy():.2f}")
print(f"  loss  = (y_hat - y)² = {loss.numpy():.4f}")
print(f"  ∂loss/∂w (automático)  = {grad.numpy():.4f}")
print(f"  ∂loss/∂w (manual) = 2(wx-y)x = {2*(w.numpy()*x.numpy()-y.numpy())*x.numpy():.4f}")
print("  → ¡coinciden! TF calculó la derivada sin que nosotros escribiéramos nada.")

---
## 3. API Sequential: construir modelos

In [ ]:
# API Sequential: el modo más sencillo de crear modelos en Keras

# Modelo para clasificación binaria (como el de la Clase 3, pero en 3 líneas)
modelo = keras.Sequential([
    layers.Input(shape=(2,)),            # capa de entrada: define la forma
    layers.Dense(8, activation='relu'),  # capa oculta: 8 neuronas, ReLU
    layers.Dense(4, activation='relu'),  # capa oculta: 4 neuronas, ReLU
    layers.Dense(1, activation='sigmoid'), # salida: 1 neurona, Sigmoid (binario)
])

# Resumen del modelo
modelo.summary()

In [ ]:
# Capas más importantes de Keras
print("Tipos de capas en Keras:")
print("""
layers.Dense(units, activation)     → capa totalmente conectada
layers.Input(shape)                 → define la forma de la entrada
layers.Dropout(rate)                → regularización (Clase 6)
layers.BatchNormalization()         → normalización por lote (Clase 6)
layers.Conv2D(filters, kernel_size) → convolución 2D (Clase 7)
layers.MaxPooling2D(pool_size)      → pooling (Clase 7)
layers.Flatten()                    → aplanar tensor a 1D
layers.LSTM(units)                  → capa recurrente (Clase 8)
layers.Embedding(vocab, dim)        → embeddings de texto (Clase 8)

Activaciones: 'relu', 'sigmoid', 'tanh', 'softmax', 'linear'
""")

# Varios modelos de ejemplo
print("Modelo multiclase (3 clases):")
modelo_multi = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(3,  activation='softmax'),  # softmax para multiclase
])
modelo_multi.summary()

print("\nModelo de regresión:")
modelo_reg = keras.Sequential([
    layers.Input(shape=(5,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),  # sin activación = lineal → para regresión
])
modelo_reg.summary()

---
## 4. Compilar y entrenar

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Datos
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_n = sc.fit_transform(X_tr)
X_te_n = sc.transform(X_te)

# 1. Crear modelo
tf.random.set_seed(42)
modelo = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8,  activation='relu'),
    layers.Dense(1,  activation='sigmoid'),
])

# 2. Compilar: definir optimizer, loss y métricas
modelo.compile(
    optimizer='adam',                          # o: keras.optimizers.Adam(lr=0.001)
    loss='binary_crossentropy',                # BCE para clasificación binaria
    metrics=['accuracy']                       # métricas a monitorizar
)

# 3. Entrenar
historial = modelo.fit(
    X_tr_n, y_tr,
    epochs=50,             # número de épocas
    batch_size=32,         # tamaño del mini-batch
    validation_split=0.2,  # 20% del train como validación
    verbose=1              # 1: mostrar progreso, 0: silencioso
)

In [ ]:
# Visualizar el historial de entrenamiento
def plot_historial(historial, titulo=''):
    hist = historial.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss
    axes[0].plot(hist['loss'],     'steelblue', lw=2, label='Train loss')
    axes[0].plot(hist['val_loss'], 'coral',     lw=2, label='Val loss')
    axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Pérdida{" — " + titulo if titulo else ""}')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(hist['accuracy'],     'steelblue', lw=2, label='Train acc')
    axes[1].plot(hist['val_accuracy'], 'coral',     lw=2, label='Val acc')
    axes[1].set_xlabel('Época'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'Accuracy{" — " + titulo if titulo else ""}')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

plot_historial(historial, 'Make Moons')

---
## 5. Evaluar, predecir y guardar

In [ ]:
# Evaluar en test
loss_te, acc_te = modelo.evaluate(X_te_n, y_te, verbose=0)
print(f"Test  → Loss: {loss_te:.4f}, Accuracy: {acc_te:.4f}")

# Predecir probabilidades
probas = modelo.predict(X_te_n[:5], verbose=0)
clases = (probas >= 0.5).astype(int).ravel()

print(f"\nPrimeras 5 muestras del test:")
print(f"  Prob clase 1: {probas.ravel()[:5].round(4)}")
print(f"  Clase predicha: {clases}")
print(f"  Clase real:     {y_te[:5]}")

# Frontera de decisión
x1_range = np.linspace(X_te_n[:,0].min()-0.5, X_te_n[:,0].max()+0.5, 200)
x2_range = np.linspace(X_te_n[:,1].min()-0.5, X_te_n[:,1].max()+0.5, 200)
XX, YY = np.meshgrid(x1_range, x2_range)
Z = modelo.predict(np.c_[XX.ravel(), YY.ravel()], verbose=0).reshape(XX.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(XX, YY, Z, alpha=0.3, cmap='RdBu')
ax.contour(XX, YY, Z, levels=[0.5], colors='black', linewidths=2)
c_map = ['coral' if yi==0 else 'steelblue' for yi in y_te]
ax.scatter(X_te_n[:,0], X_te_n[:,1], c=c_map, s=25, edgecolors='black', lw=0.5, zorder=5)
ax.set_title(f'Frontera de decisión — Keras (acc={acc_te:.2%})')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
plt.tight_layout(); plt.show()

In [ ]:
# Guardar y cargar modelos
# Formato recomendado: SavedModel (carpeta) o .keras
modelo.save('modelo_moons.keras')
print("Modelo guardado como 'modelo_moons.keras'")

# Cargar
modelo_cargado = keras.models.load_model('modelo_moons.keras')
loss_c, acc_c = modelo_cargado.evaluate(X_te_n, y_te, verbose=0)
print(f"Modelo cargado → Accuracy: {acc_c:.4f}  (igual que antes)")

# Solo guardar los pesos
modelo.save_weights('pesos_moons.weights.h5')
print("Pesos guardados como 'pesos_moons.weights.h5'")

In [ ]:
# Callbacks: acciones automáticas durante el entrenamiento
tf.random.set_seed(42)

callbacks = [
    # EarlyStopping: detener cuando val_loss no mejora
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,         # esperar 10 épocas sin mejora antes de parar
        restore_best_weights=True  # restaurar los mejores pesos
    ),
    # ModelCheckpoint: guardar el mejor modelo
    keras.callbacks.ModelCheckpoint(
        filepath='mejor_modelo.keras',
        monitor='val_loss',
        save_best_only=True
    ),
    # ReduceLROnPlateau: reducir lr si se estanca
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,       # multiplicar lr × 0.5
        patience=5,
        min_lr=1e-6
    ),
]

modelo2 = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8,  activation='relu'),
    layers.Dense(1,  activation='sigmoid'),
])
modelo2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

historial2 = modelo2.fit(
    X_tr_n, y_tr,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=0
)

n_epocas = len(historial2.history['loss'])
loss_te2, acc_te2 = modelo2.evaluate(X_te_n, y_te, verbose=0)
print(f"Entrenó {n_epocas} épocas (de 200 posibles) — EarlyStopping activó")
print(f"Test accuracy: {acc_te2:.4f}")
plot_historial(historial2, 'con EarlyStopping')

---
## Práctica

### Ejercicio 1 — Primer modelo Keras en MNIST

**Enunciado:** MNIST es el 'Hello World' del Deep Learning: 70000 imágenes de dígitos escritos a mano.

```python
(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
# X: (N, 28, 28), y: (N,) con clases 0-9
```

1. Normaliza: `X = X / 255.0` y aplana: `X = X.reshape(-1, 784)`
2. Construye un modelo Sequential: 784 → 128 → 64 → 10 (softmax)
3. Compila: `optimizer='adam'`, `loss='sparse_categorical_crossentropy'`
4. Entrena 10 épocas, batch=64
5. ¿Qué accuracy consigue en test?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784) / 255.0
X_te = X_te.reshape(-1, 784) / 255.0
print(f"Train: {X_tr.shape}, Test: {X_te.shape}")

tf.random.set_seed(42)
modelo = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),  # 10 clases → softmax
])

modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # y son enteros (no one-hot)
    metrics=['accuracy']
)
modelo.summary()

hist = modelo.fit(X_tr, y_tr, epochs=10, batch_size=64,
                  validation_split=0.1, verbose=1)

loss, acc = modelo.evaluate(X_te, y_te, verbose=0)
print(f"\nTest accuracy: {acc:.4f} ({acc:.2%})")

# Ver algunas predicciones
y_pred = modelo.predict(X_te[:9], verbose=0).argmax(axis=1)
fig, axes = plt.subplots(3, 3, figsize=(7, 7))
for ax, i in zip(axes.ravel(), range(9)):
    ax.imshow(X_te[i].reshape(28,28), cmap='gray')
    ok = y_pred[i] == y_te[i]
    ax.set_title(f'Pred:{y_pred[i]}  Real:{y_te[i]}',
                 color='green' if ok else 'red')
    ax.axis('off')
plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 2 — Efecto de la arquitectura

**Enunciado:** Con el dataset MNIST, compara 3 arquitecturas distintas:

- Modelo A: 784 → 32 → 10 (pequeño)
- Modelo B: 784 → 256 → 128 → 10 (mediano)
- Modelo C: 784 → 512 → 256 → 128 → 64 → 10 (grande)

1. Entrena cada uno 5 épocas
2. Compara: accuracy test, tiempo de entrenamiento, nº de parámetros
3. ¿Cuál tiene la mejor relación calidad/coste?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784) / 255.0
X_te = X_te.reshape(-1, 784) / 255.0

arquitecturas = [
    ('A (pequeño)',  [32]),
    ('B (mediano)', [256, 128]),
    ('C (grande)',  [512, 256, 128, 64]),
]

print(f"{'Modelo':15} {'Params':>10} {'Acc test':>10} {'Tiempo (s)':>12}")
print("-" * 52)

for nombre, capas_ocultas in arquitecturas:
    tf.random.set_seed(42)
    capas = [layers.Input(shape=(784,))]
    for n in capas_ocultas:
        capas.append(layers.Dense(n, activation='relu'))
    capas.append(layers.Dense(10, activation='softmax'))

    m = keras.Sequential(capas)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

    t0 = time.time()
    m.fit(X_tr, y_tr, epochs=5, batch_size=64, verbose=0)
    t1 = time.time()

    _, acc = m.evaluate(X_te, y_te, verbose=0)
    params = m.count_params()
    print(f"{nombre:15} {params:>10,} {acc:>10.4f} {t1-t0:>12.1f}")
```
</details>

---
### Ejercicio 3 — Callbacks en acción

**Enunciado:** Entrena un modelo con EarlyStopping y ReduceLROnPlateau:

1. Dataset: `keras.datasets.fashion_mnist` (ropa en lugar de dígitos)
2. Arquitectura: 784 → 256 → 128 → 10 (softmax)
3. EarlyStopping(patience=8, restore_best_weights=True)
4. ReduceLROnPlateau(factor=0.5, patience=4)
5. Plotea la curva de loss y señala dónde redujo el lr

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.reshape(-1, 784) / 255.0
X_te = X_te.reshape(-1, 784) / 255.0

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

lr_hist = []
class LRLogger(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr_hist.append(float(self.model.optimizer.learning_rate))

cbs = [
    keras.callbacks.EarlyStopping('val_loss', patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=4, min_lr=1e-6),
    LRLogger(),
]

hist = m.fit(X_tr, y_tr, epochs=100, batch_size=64, validation_split=0.1,
             callbacks=cbs, verbose=0)

_, acc = m.evaluate(X_te, y_te, verbose=0)
print(f"Épocas entrenadas: {len(hist.history['loss'])}")
print(f"Test accuracy: {acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist.history['loss'],     label='Train loss')
axes[0].plot(hist.history['val_loss'], label='Val loss')
# Señalar cambios de lr
for i in range(1, len(lr_hist)):
    if lr_hist[i] < lr_hist[i-1]:
        axes[0].axvline(i, color='red', ls='--', alpha=0.6)
axes[0].set_title('Loss (líneas rojas = reducción lr)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(lr_hist, 'orange', lw=2)
axes[1].set_title('Learning rate por época')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('lr')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 4 — Regresión con Keras

**Enunciado:** Usa California Housing con Keras:

```python
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()
X, y = data.data, data.target
```

1. Normaliza X con StandardScaler
2. Modelo: 8 → 64 → 32 → 1 (sin activación final)
3. Compila con `loss='mse'`, `metrics=['mae']`
4. Entrena con EarlyStopping
5. Reporta MAE en test (en 100k $)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = fetch_california_housing()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

sc = StandardScaler()
X_tr_n = sc.fit_transform(X_tr)
X_te_n = sc.transform(X_te)

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(8,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),   # sin activación → regresión
])
m.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse', metrics=['mae'])

hist = m.fit(
    X_tr_n, y_tr, epochs=100, batch_size=64, validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True)],
    verbose=0
)

mse, mae = m.evaluate(X_te_n, y_te, verbose=0)
print(f"Test MAE: {mae:.4f} (100k $) = {mae*100:.0f}k $ de error medio")
print(f"Test RMSE: {mse**0.5:.4f}")
```
</details>

---
### Ejercicio 5 (reto) — API Funcional de Keras

**Enunciado:** La API Funcional permite crear arquitecturas más complejas. Construye un modelo con dos ramas que se fusionan:

```
Entrada (10 features)
       ↓                ↓
Dense(16, relu)    Dense(8, relu)
       ↓                ↓
Dense(8, relu)     Dense(4, relu)
       └──── Concatenar ────┘
              ↓
         Dense(1, sigmoid)
```

1. Usa `keras.Input`, `layers.Dense`, `layers.Concatenate`, `keras.Model`
2. Prueba en make_classification(n_features=10)
3. Compara con un modelo Sequential de arquitectura similar

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(1000, n_features=10, n_informative=6, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, 0.2, random_state=42)
sc = StandardScaler()
X_tr_n = sc.fit_transform(X_tr)
X_te_n = sc.transform(X_te)

# API Funcional
tf.random.set_seed(42)
entrada = keras.Input(shape=(10,))

# Rama 1
rama1 = layers.Dense(16, activation='relu')(entrada)
rama1 = layers.Dense(8,  activation='relu')(rama1)

# Rama 2
rama2 = layers.Dense(8,  activation='relu')(entrada)
rama2 = layers.Dense(4,  activation='relu')(rama2)

# Fusionar
fusionado = layers.Concatenate()([rama1, rama2])
salida    = layers.Dense(1, activation='sigmoid')(fusionado)

modelo_funcional = keras.Model(inputs=entrada, outputs=salida, name='modelo_dos_ramas')
modelo_funcional.summary()

modelo_funcional.compile('adam', 'binary_crossentropy', ['accuracy'])
hist = modelo_funcional.fit(X_tr_n, y_tr, epochs=50, batch_size=32,
                            validation_split=0.2, verbose=0)
_, acc = modelo_funcional.evaluate(X_te_n, y_te, verbose=0)
print(f"Modelo dos ramas → Test accuracy: {acc:.4f}")

# Comparar con Sequential
tf.random.set_seed(42)
m_seq = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8,  activation='relu'),
    layers.Dense(1,  activation='sigmoid'),
])
m_seq.compile('adam', 'binary_crossentropy', ['accuracy'])
m_seq.fit(X_tr_n, y_tr, epochs=50, batch_size=32, validation_split=0.2, verbose=0)
_, acc_seq = m_seq.evaluate(X_te_n, y_te, verbose=0)
print(f"Modelo Sequential → Test accuracy: {acc_seq:.4f}")
```
</details>

---
## Resumen

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Construir
modelo = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(n_clases, activation='softmax'),  # o 1+sigmoid para binario
])

# 2. Compilar
modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # multiclase con enteros
    # loss='categorical_crossentropy'        # multiclase con one-hot
    # loss='binary_crossentropy'             # binario
    # loss='mse'                             # regresión
    metrics=['accuracy']
)

# 3. Entrenar
hist = modelo.fit(X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2)

# 4. Evaluar y predecir
modelo.evaluate(X_te, y_te)
modelo.predict(X_nuevo)

# 5. Guardar
modelo.save('mi_modelo.keras')
modelo = keras.models.load_model('mi_modelo.keras')

# Callbacks útiles
keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
keras.callbacks.ModelCheckpoint('mejor.keras', save_best_only=True)
```

## ¿Qué viene en la próxima clase?
En la **Clase 5** construiremos **redes densas** más completas para clasificación: MNIST en detalle, visualización de lo que aprende cada neurona, y comparación de distintas configuraciones.